In [0]:

bronze_file_path = (
    "/Volumes/workspace/default/bookshelf_raw_volume/"
    "bronze/open_library/"
    "books_data_20261111T133441Z.json"
)

raw_df = (
    spark.read
    .option("multiLine", "true")
    .json(bronze_file_path)
)

print("Bronze data schema:")
raw_df.printSchema()

print("Number of raw response records:", raw_df.count())


In [0]:
from pyspark.sql.functions import col, explode

books_df = (
    raw_df
    .select(explode(col("docs")).alias("book"))
    .select(
        col("book.key").alias("work_id"),
        col("book.title").alias("title"),
        col("book.author_name").alias("authors"),
        col("book.first_publish_year").alias("first_publish_year"),
        col("book.edition_count").alias("edition_count")
    )
)

print("Number of extracted books:", books_df.count())

display(books_df.limit(10))

In [0]:

from pyspark.sql.functions import col, trim

# 1. Remove leading/trailing spaces from identifiers and titles
cleaned_df = (
    books_df
    .withColumn("work_id", trim(col("work_id")))
    .withColumn("title", trim(col("title")))
)

# 2. Remove records without a valid work ID or title
cleaned_df = cleaned_df.filter(
    col("work_id").isNotNull() &
    (col("work_id") != "") &
    col("title").isNotNull() &
    (col("title") != "")
)

# 3. Remove duplicate books using their Open Library work ID
cleaned_df = cleaned_df.dropDuplicates(["work_id"])

# 4. Check the cleaned dataset
print("Books before cleaning:", books_df.count())
print("Books after cleaning:", cleaned_df.count())

print("Cleaned data schema:")
cleaned_df.printSchema()

display(cleaned_df.limit(10))


In [0]:

from pyspark.sql.functions import (
    col, count, when, min, max
)

# Count missing values in each column
print("Missing value report:")

cleaned_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in cleaned_df.columns
]).show()

# Check publication year range
print("Publication year range:")

cleaned_df.select(
    min("first_publish_year").alias("earliest_year"),
    max("first_publish_year").alias("latest_year")
).show()

# Check invalid edition counts
invalid_editions = cleaned_df.filter(
    (col("edition_count") <= 0) |
    col("edition_count").isNull()
).count()

print("Records with missing or invalid edition counts:", invalid_editions)

# Check invalid publication years
invalid_years = cleaned_df.filter(
    (col("first_publish_year") <= 0) |
    col("first_publish_year").isNull()
).count()

print("Records with missing or invalid publication years:", invalid_years)


In [0]:

# Silver Delta table name
silver_table = "workspace.default.books_silver"

# Save the cleaned DataFrame as a Delta table
(
    cleaned_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print("Silver table saved successfully:", silver_table)

# Verify the saved table
silver_df = spark.table(silver_table)

print("Records in Silver table:", silver_df.count())

display(silver_df.limit(10))
